# Test the saved whale CNN (`models/whale_cnn_v1.pt`)

Self-contained: nothing here imports from `train.py`. Cells 1–2 load the checkpoint; 3 evaluates on the held-out test split;
4 runs the model on any WAV file; 5 checks the ONNX export gives the same answer.

In [ ]:
import json, pathlib
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt

MODEL = pathlib.Path("models/whale_cnn_v2.pt")      # or models/whale_cnn_v1.pt
FEATURES = None   # None = read the feature folder the checkpoint was trained on; only needed for the test-split cells
DEV = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

## 1. The architecture (must match what was trained)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                                 nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True), nn.MaxPool2d(2))
    def forward(self, x): return self.net(x)

class WhaleCNN(nn.Module):
    def __init__(self, n_classes, width=32, dropout=0.3):
        super().__init__(); w = width
        self.stem = nn.Sequential(nn.AvgPool2d(2), nn.Conv2d(1, w, 5, stride=2, padding=2, bias=False), nn.BatchNorm2d(w), nn.ReLU(inplace=True))
        self.features = nn.Sequential(ConvBlock(w, w), ConvBlock(w, 2*w), ConvBlock(2*w, 4*w), ConvBlock(4*w, 8*w))
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(8*w*2, n_classes))
    def forward(self, x):                                            # (B, 1, 128, 301)
        h = self.features(self.stem(x))
        return self.head(torch.cat([h.mean(dim=(2, 3)), h.amax(dim=(2, 3))], 1))

## 2. Load the checkpoint
The `.pt` carries the class list, width, feature spec and the metrics it scored at training time.

In [ ]:
ck = torch.load(MODEL, map_location="cpu")
CLASSES, spec = ck["classes"], ck["feature_spec"]
model = WhaleCNN(len(CLASSES), width=ck["arch"]["width"]); model.load_state_dict(ck["model"]); model.eval().to(DEV)
print("classes:", len(CLASSES)); print("feature spec:", spec); print("training-time metrics:", ck["metrics"])
if FEATURES is None: FEATURES = pathlib.Path(ck.get("train_config", {}).get("features", "../dataset/features/whales_32k_mel128_3s"))
print("features:", FEATURES)

## 3. Evaluate on the test split
Uses the precomputed windows + manifest from `../dataset/features/`. Skip to section 4 if you only want to try WAV files.

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix
X = np.load(FEATURES / "windows.npy", mmap_mode="r"); m = pd.read_csv(FEATURES / "manifest.csv")
RARE_TO_OTHER = {"Peponocephala_electra", "Stenella_clymene", "Stenella_frontalis", "Lagenodelphis_hosei"}   # same merge as training
if "hierarchy" not in m.columns: m.loc[m.label.isin(RARE_TO_OTHER), "label"] = "other_whale"   # v1 manifests only
CID = {c: i for i, c in enumerate(CLASSES)}; m["y"] = m.label.map(CID); m = m[m.y.notna()].copy(); m["y"] = m.y.astype(int)
te = m.index[m.split == "test"].values

preds = []
with torch.no_grad():
    for s in range(0, len(te), 256):
        xb = torch.from_numpy(np.asarray(X[te[s:s+256]], dtype=np.float32))[:, None].to(DEV)
        preds.append(model(xb).argmax(1).cpu())
p = torch.cat(preds).numpy(); y = m.y.values[te]
print(f"test windows: {len(te)}   macro-F1 {f1_score(y, p, average='macro'):.3f}   accuracy {(p == y).mean():.3f}")
present = sorted(set(y) | set(p))
print(classification_report(y, p, labels=present, target_names=[CLASSES[i] for i in present], zero_division=0))

In [ ]:
cm = confusion_matrix(y, p, labels=list(range(len(CLASSES))), normalize="true")
plt.figure(figsize=(11, 9)); plt.imshow(cm, cmap="Blues", vmin=0, vmax=1)
plt.xticks(range(len(CLASSES)), CLASSES, rotation=90); plt.yticks(range(len(CLASSES)), CLASSES)
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("test confusion (row-normalized)"); plt.colorbar(); plt.tight_layout(); plt.show()

## 4. Classify a WAV file
Re-creates the training feature pipeline from `spec`: mono → resample → 3 s windows (50 % overlap) → log-mel → per-window z-score.
Point `wav` at anything: files under `../dataset/audio/`, or your own hydrophone recordings.

In [ ]:
import soundfile as sf, librosa

def windows_from_wav(path, spec):
    y, sr0 = sf.read(str(path), dtype="float32", always_2d=True); y = y.mean(1); sr = int(spec["sr"])
    if sr0 != sr: y = librosa.resample(y, orig_sr=sr0, target_sr=sr, res_type="soxr_hq")
    n = int(spec["win_s"] * sr); hop = n // 2
    if len(y) < n: pad = n - len(y); y = np.pad(y, (pad // 2, pad - pad // 2)); starts = [0]
    else: starts = list(range(0, len(y) - n + 1, hop))
    T = 1 + n // int(spec["hop"]); out = np.zeros((len(starts), int(spec["n_mels"]), T), dtype=np.float32)
    for k, s in enumerate(starts):
        mel = librosa.feature.melspectrogram(y=y[s:s+n], sr=sr, n_fft=int(spec["n_fft"]), hop_length=int(spec["hop"]),
                                             n_mels=int(spec["n_mels"]), fmin=spec["fmin"], fmax=spec["fmax"])
        z = np.log1p(mel); z = (z - z.mean()) / (z.std() + 1e-6); out[k, :, :z.shape[1]] = z[:, :T]
    return out, np.array(starts) / sr

def classify(path, top=3, plot=True):
    W, t = windows_from_wav(path, spec)
    with torch.no_grad(): probs = torch.softmax(model(torch.from_numpy(W)[:, None].to(DEV)), 1).cpu().numpy()
    clip = probs.mean(0); order = np.argsort(-clip)[:top]
    print(pathlib.Path(path).name, f"({len(W)} windows)")
    for i in order: print(f"   {CLASSES[i]:32s} {clip[i]:.2f}")
    if plot:
        fig, ax = plt.subplots(1, 2, figsize=(11, 3), gridspec_kw={"width_ratios": [2, 1]})
        ax[0].imshow(W[0], origin="lower", aspect="auto", cmap="magma", vmin=-2, vmax=3); ax[0].set_title("first window (log-mel, z-scored)")
        ax[1].barh([CLASSES[i] for i in order[::-1]], clip[order[::-1]]); ax[1].set_xlim(0, 1); ax[1].set_title("clip-level probability"); plt.tight_layout(); plt.show()
    return probs, t

audio = pathlib.Path("../dataset/audio")
examples = [next((audio / d).rglob("*.wav")) for d in ["orcasound/SRKW_call", "cornell_whale/upcall", "watkins/Physeter_macrocephalus", "watkins/Megaptera_novaeangliae"] if (audio / d).exists()]
for wav in examples: classify(wav)

In [ ]:
# your own file:
# wav = "/path/to/recording.wav"
# probs, t = classify(wav)
# per-window timeline for a long recording:
# for ti, pi in zip(t, probs): print(f"{ti:7.1f}s  {CLASSES[pi.argmax()]:30s} {pi.max():.2f}")

## 5. ONNX export agrees with PyTorch?
Useful if you plan to run the model with ONNX Runtime on the buoy.

In [ ]:
try:
    import onnxruntime as ort
    sess = ort.InferenceSession("models/whale_cnn_v1.onnx", providers=["CPUExecutionProvider"])
    W, _ = windows_from_wav(examples[0], spec)
    o = sess.run(["logits"], {"log_mel": W[:4, None]})[0]
    with torch.no_grad(): pt = model(torch.from_numpy(W[:4])[:, None].to(DEV)).cpu().numpy()
    print("max |onnx - torch| =", float(np.abs(o - pt).max()), "| argmax agree:", bool((o.argmax(1) == pt.argmax(1)).all()))
except ImportError:
    print("pip install onnxruntime to run this cell")